In [1]:
import pandas as pd
import numpy as np
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer

In [2]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')
device

'cuda'

In [3]:
df = pd.read_csv("C:/AEGIS/roadmap/Deep_Learning/customer_churn/data/Churn_Modelling.csv")
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
df.shape

(10000, 14)

In [5]:
df = df.drop(columns = ['Surname', 'RowNumber', 'CustomerId'])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 859.5 KB


In [7]:
X = df.drop(columns='Exited')
y = df['Exited'].values

In [8]:
num_cols = X.select_dtypes(include='number').columns.tolist()
num_cols

['CreditScore',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary']

In [9]:
cat_cols = X.select_dtypes(include='str').columns.tolist()
cat_cols

['Geography', 'Gender']

In [10]:
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), num_cols),
    ('ohe', OneHotEncoder(sparse_output=False, drop='first'), cat_cols)
], remainder='passthrough')

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [12]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [13]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [14]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [15]:
X_train.shape[0]

8000

In [16]:
X_train.shape[1]

11

In [17]:
len(train_dataset)

8000

In [18]:
train_loader = DataLoader(train_dataset, batch_size = 1000, shuffle = True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size= 1000, shuffle=False, pin_memory=True)

In [19]:
class ANN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 5),
            nn.ReLU(),
            nn.Linear(5, 3),
            nn.ReLU(),
            nn.Linear(3,output_dim)
        )

    def forward(self, x):
        return self.model(x)

In [20]:
learning_rate = 0.5
epochs = 100

In [21]:
model = ANN(X_train.shape[1], 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr = learning_rate)

In [22]:
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs , batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1} : Loss {avg_loss}")

Epoch 1 : Loss 0.5590362809598446
Epoch 2 : Loss 0.5032815858721733
Epoch 3 : Loss 0.4959247224032879
Epoch 4 : Loss 0.4860699437558651
Epoch 5 : Loss 0.47574392706155777
Epoch 6 : Loss 0.4649530500173569
Epoch 7 : Loss 0.45424820110201836
Epoch 8 : Loss 0.4449317716062069
Epoch 9 : Loss 0.43613941222429276
Epoch 10 : Loss 0.42890722304582596
Epoch 11 : Loss 0.4218587651848793
Epoch 12 : Loss 0.415602121502161
Epoch 13 : Loss 0.40883251279592514
Epoch 14 : Loss 0.40160205587744713
Epoch 15 : Loss 0.39438943937420845
Epoch 16 : Loss 0.38902772590518
Epoch 17 : Loss 0.3833119161427021
Epoch 18 : Loss 0.37774229794740677
Epoch 19 : Loss 0.3735411688685417
Epoch 20 : Loss 0.370404876768589
Epoch 21 : Loss 0.3660259246826172
Epoch 22 : Loss 0.36189479753375053
Epoch 23 : Loss 0.35907330736517906
Epoch 24 : Loss 0.35873936489224434
Epoch 25 : Loss 0.35641423240303993
Epoch 26 : Loss 0.3546576164662838
Epoch 27 : Loss 0.35258759185671806
Epoch 28 : Loss 0.3509293459355831
Epoch 29 : Loss 0.35

In [23]:
model.eval()

ANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=3, bias=True)
    (3): ReLU()
    (4): Linear(in_features=3, out_features=2, bias=True)
  )
)

In [24]:
y_preds = []
y_true = []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        outputs = model(batch_features)
        preds = torch.argmax(outputs, dim=1)
        y_preds.extend(preds.cpu().numpy())
        y_true.extend(batch_labels.numpy())

print(classification_report(y_true, y_preds))

              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1607
           1       0.70      0.52      0.60       393

    accuracy                           0.86      2000
   macro avg       0.80      0.73      0.76      2000
weighted avg       0.85      0.86      0.85      2000

